# Osonye Onyemazuwa — Ad Spend Data
### Day 1-2: Created main branch README + Setup issues on Github + Acquire and load Ad Spend dataset

Data file: `ad_spend.csv` (place in a local `data/` folder, excluded via `.gitignore`)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs("cleaned", exist_ok=True)
os.makedirs("charts", exist_ok=True)

df = pd.read_csv("ad_spend.csv")
print(df.shape)
df.head()


## Initial structure check

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

In [ ]:
df.describe(include='all').T

In [ ]:
# Convert date from string to a real datetime type
df['date'] = pd.to_datetime(df['date'])
df.dtypes

## Quick look: spend by channel

In [ ]:
df.groupby('channel')['ad_spend'].agg(['sum', 'mean', 'count']).sort_values('sum', ascending=False)


## Quick look: date range

In [ ]:
print("Earliest:", df['date'].min())
print("Latest:", df['date'].max())


## Notes (local analysis log)

- Rows: 2,599, no missing values in any column (confirmed clean on delivery)
- Date range: 2021-01-20 to 2024-01-06
- `campaign_id` is numeric, joins cleanly to `campaigns.csv` (all 50 IDs used, no orphans either direction)
- No duplicate `spend_id` rows
- Spend by channel: **Affiliate (\$6,455)** > **Paid Search (\$6,151)** > **Email (\$5,614)** \> **Display (\$4,480)** \> **Social (\$4,176)**

Team-facing decisions/blockers from this analysis are tracked in the repo README, not duplicated here.


---
### Next steps (Day 3 — Issue #3)
- Standardize `channel` / `utm_source` casing to match `events.traffic_source`
- Decide with team how to handle Affiliate/Display spend attribution
- Check for duplicate `spend_id` rows


---
## Day 3 — Clean campaign IDs and standardize channel naming
**Issue #8:** Clean missing UTM parameters and inconsistent campaign IDs

Checked already:
- `campaign_id`: all 50 IDs match `campaigns.csv` exactly (no orphans either direction) — no cleaning needed here
- `spend_id`: no duplicates — no cleaning needed here
- No missing values anywhere in this file

Remaining real issue: `channel` values here don't match `events.traffic_source`
casing/naming, which will break the Week 2 SQL joins if not fixed now.


In [ ]:
# Confirm campaign_id integrity (already checked, keeping as a documented test)
campaigns = pd.read_csv("campaigns.csv")

ad_ids = set(df['campaign_id'])
valid_ids = set(campaigns['campaign_id'])

orphans_in_spend = ad_ids - valid_ids
unused_campaigns = valid_ids - ad_ids

print("campaign_ids in ad_spend but not in campaigns.csv:", orphans_in_spend)
print("campaign_ids in campaigns.csv but never spent on:", unused_campaigns)
assert len(orphans_in_spend) == 0, "Found orphan campaign_ids -- investigate before Week 2"

In [ ]:
# Confirm no duplicate spend_id rows
dupes = df['spend_id'].duplicated().sum()
print(f"Duplicate spend_id rows: {dupes}")
assert dupes == 0, "Found duplicate spend_id rows -- dedupe before proceeding"

### Standardize `channel` for downstream joins

`events.traffic_source` uses: Direct / Email / Organic / Paid Search / Social
(in multiple casings). `ad_spend.channel` uses: Affiliate / Display / Email /
Paid Search / Social.

Standardizing to lowercase + underscores now so this matches whatever
Member B does to `events.traffic_source` in their own cleaning step —
**worth confirming the exact convention with Member B before Day 4**,
so both sides land on the same format.

In [ ]:
df['channel_clean'] = df['channel'].str.strip().str.lower().str.replace(' ', '_', regex=False)
df['utm_source_clean'] = df['utm_source'].str.strip().str.lower()

print(df['channel_clean'].unique())
print(df['utm_source_clean'].unique())

### Flag: Affiliate / Display have no equivalent in `events.traffic_source`

> This isn't something Day 3 can "clean away" — it's a team decision on how
> Affiliate and Display spend gets attributed (map to an existing category,
> or keep as a separate "unattributed spend" line). Documented in README;
> raising with team before Week 3 Fact table work.

In [ ]:
# Save cleaned output locally (not committed -- .gitignore covers this)
df.to_csv("cleaned/ad_spend_clean.csv", index=False)
print("Saved cleaned/ad_spend_clean.csv:", df.shape)

---
## Day 4 — EDA on Ad Spend: spend by channel/campaign/day
**Issue #9:** EDA on Ad Spend: spend by channel/campaign/day; commit charts + notes.

In [ ]:
df.head()

### Spend by channel

In [ ]:
spend_by_channel = df.groupby('channel_clean')['ad_spend'].sum().sort_values(ascending=False)
spend_by_channel

In [ ]:
spend_by_channel.plot(kind='bar', figsize=(8,4), title='Total Ad Spend by Channel')
plt.ylabel('Spend ($)')
plt.tight_layout()
plt.xticks(rotation=0)
plt.savefig('charts/spend_by_channel.png')
plt.show()

### Spend by campaign (top 10)

In [ ]:
spend_by_campaign = df.groupby('campaign_id')['ad_spend'].sum().sort_values(ascending=False)
spend_by_campaign.head(10)

In [ ]:
spend_by_campaign.head(10).plot(kind='bar', figsize=(8,4), title='Top 10 Campaigns by Spend')
plt.ylabel('Spend ($)')
plt.xlabel('Campaign ID')
plt.tight_layout()
plt.xticks(rotation=0)
plt.savefig('charts/top10_campaigns.png')
plt.show()

### Spend over time (daily)

In [ ]:
spend_by_day = df.groupby(df['date'].dt.date)['ad_spend'].sum()
spend_by_day.describe()

In [ ]:
spend_by_day.plot(figsize=(10,4), title='Daily Ad Spend Over Time')
plt.ylabel('Spend ($)')
plt.xlabel('Date')
plt.tight_layout()
plt.savefig('charts/daily_spend_trend.png')
plt.show()

### Day 4 notes

> - Highest-spend channel: Affiliate; lowest: Social
> - Campaign #48 is the single highest-spend campaign **(\$1,001.70)** — worth checking
>  if it's genuinely a bigger campaign or a possible outlier for Day 5
> - Daily spend ranges from **\$10** (single min-spend day) to **\$78.36** (busiest day), averaging **~\$27.79/day** across 967 active spend-days
> - Clear cyclical pattern — spend alternates between low-baseline periods **(~$10/day)** and higher bursts every few months, rather than being random. Worth checking in Day 5 whether these bursts align with specific campaigns.csv start/end dates.

---
## Day 5 — Identify and flag outlier/suspicious spend entries
**Issue #10:** Identify and flag outlier/suspicious spend entries.


### First pass: revisit Campaign #48 (yesterday's flag)

In [ ]:
campaign_48 = df[df['campaign_id'] == 48]
print("Campaign #48 rows:", len(campaign_48))
print("Total spend:", campaign_48['ad_spend'].sum())
campaign_48[['date', 'ad_spend', 'clicks', 'impressions']].describe()

> **Finding:** Campaign #48's high total **(\$1,001.70)** is NOT a single suspicious
> entry — it's 90 separate spend rows (Jan-Apr 2022), each mostly at the \~$10
> baseline. It's simply the longest-running campaign in the dataset, not an
> anomaly. Clearing this flag from Day 4.

### Structural sanity checks

In [ ]:
# Zero clicks but positive spend (would indicate wasted/broken spend)
zero_clicks = df[(df['clicks'] == 0) & (df['ad_spend'] > 0)]
print("Rows with 0 clicks but spend > 0:", len(zero_clicks))

# Duplicate campaign_id + date combos (possible double-charge)
dupe_combo = df[df.duplicated(subset=['campaign_id', 'date'], keep=False)]
print("Rows sharing same campaign_id + date:", len(dupe_combo))

# Negative spend / zero impressions
print("Negative ad_spend rows:", (df['ad_spend'] < 0).sum())
print("Zero impressions rows:", (df['impressions'] == 0).sum())

> All clean — no zero-click spend, no duplicate campaign+date charges, no
> negative values, no zero-impression rows. Confirms Day 2's finding that
> this dataset is unusually clean.

### Why a plain IQR check is misleading here

In [ ]:
Q1 = df['ad_spend'].quantile(0.25)
Q3 = df['ad_spend'].quantile(0.75)
print("Q1:", Q1, "  Q3:", Q3)

> Q1 and Q3 are both **\$10** — spend is heavily clustered at a **\$10** floor, so a
standard IQR bound collapses to basically zero width. That would flag 220
rows (~8.5% of data) as "outliers" just for spending slightly above $10,
which isn't actually suspicious. Using **implied CPC (spend / clicks)**
instead is a more meaningful signal for genuinely unusual entries.

In [ ]:
df_clicks = df[df['clicks'] > 0].copy()
df_clicks['implied_cpc'] = df_clicks['ad_spend'] / df_clicks['clicks']
df_clicks['implied_cpc'].describe()

In [ ]:
top_cpc = df_clicks.sort_values('implied_cpc', ascending=False).head(10)
top_cpc[['spend_id', 'date', 'campaign_id', 'channel', 'ad_spend', 'clicks', 'implied_cpc']]

### Day 5 notes

> - Campaign #48 flag from Day 4 is cleared — it's a long-running campaign
  (90 days), not a single suspicious spend event
> - No structural data issues: no zero-click spend, no duplicate charges,
  no negatives, no zero impressions
> - Plain IQR is the wrong outlier method for this data (spend clusters
  tightly at a **\$10** floor) — used implied CPC instead
> - Highest implied CPC: `SP000096` (Campaign #3, 2023-10-08) at **\$17.63/click**
  — 2 clicks for $35.25. Worth a quick sanity check but not necessarily
  wrong; low click volume days naturally produce noisy CPC
> - No entries flagged as clearly erroneous/suspicious — dataset appears
  genuinely clean going into Week 2

In [ ]:
# Save flagged review list for the team (not committed -- local only)
top_cpc.to_csv("cleaned/ad_spend_flagged_for_review.csv", index=False)
print("Saved flagged review list:", top_cpc.shape)